<a href="https://colab.research.google.com/github/mohammed-fariz/project/blob/main/face_recognition_model_using_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),        # fixed size
    transforms.ToTensor(),                 # convert to tensor
    transforms.Normalize(                  # normalize
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [9]:
import os

file_path = "archive (1).zip"
print(os.path.getsize(file_path))


389784397


In [10]:
import zipfile

zip_path = "/content/archive (1).zip"
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")


ZIP extracted successfully!


In [18]:
dataset = datasets.ImageFolder(
    "/content/dataset/105_classes_pins_dataset",
    transform=transform
)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [19]:
for i, (images, labels) in enumerate(loader):
    print(f"Batch {i}:", images.shape, labels.shape)
    if i == 2:
        break



Batch 0: torch.Size([32, 3, 224, 224]) torch.Size([32])
Batch 1: torch.Size([32, 3, 224, 224]) torch.Size([32])
Batch 2: torch.Size([32, 3, 224, 224]) torch.Size([32])


In [20]:
print("Total images:", len(dataset))
print("Classes:", dataset.classes)
print("Number of classes:", len(dataset.classes))


Total images: 17534
Classes: ['pins_Adriana Lima', 'pins_Alex Lawther', 'pins_Alexandra Daddario', 'pins_Alvaro Morte', 'pins_Amanda Crew', 'pins_Andy Samberg', 'pins_Anne Hathaway', 'pins_Anthony Mackie', 'pins_Avril Lavigne', 'pins_Ben Affleck', 'pins_Bill Gates', 'pins_Bobby Morley', 'pins_Brenton Thwaites', 'pins_Brian J. Smith', 'pins_Brie Larson', 'pins_Chris Evans', 'pins_Chris Hemsworth', 'pins_Chris Pratt', 'pins_Christian Bale', 'pins_Cristiano Ronaldo', 'pins_Danielle Panabaker', 'pins_Dominic Purcell', 'pins_Dwayne Johnson', 'pins_Eliza Taylor', 'pins_Elizabeth Lail', 'pins_Emilia Clarke', 'pins_Emma Stone', 'pins_Emma Watson', 'pins_Gwyneth Paltrow', 'pins_Henry Cavil', 'pins_Hugh Jackman', 'pins_Inbar Lavi', 'pins_Irina Shayk', 'pins_Jake Mcdorman', 'pins_Jason Momoa', 'pins_Jennifer Lawrence', 'pins_Jeremy Renner', 'pins_Jessica Barden', 'pins_Jimmy Fallon', 'pins_Johnny Depp', 'pins_Josh Radnor', 'pins_Katharine Mcphee', 'pins_Katherine Langford', 'pins_Keanu Reeves', '

In [21]:
from torch.utils.data import random_split

train_size = int(0.7 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset, [train_size, val_size, test_size]
)


In [22]:


train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [28]:
import torch.nn as nn
model = nn.Sequential(
    nn.Conv2d(in_channels=3,out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 224 -> 112

    nn.Conv2d(in_channels=32,out_channels=64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 112 -> 56

    nn.Conv2d(in_channels=64,out_channels= 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2,2),    # 56 -> 28

    nn.Flatten(),
    nn.Linear(128*28*28, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 105)
).to(device)

In [29]:
# 6️⃣ Loss and Optimizer
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [37]:
num_epochs = 10

for epoch in range(num_epochs):

    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---------- VALIDATION ----------
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / val_total

    # ---------- LOG ----------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

Epoch [1/10] Train Loss: 4.6343 Train Acc: 0.0123 Val Acc: 0.0080
Epoch [2/10] Train Loss: 4.6334 Train Acc: 0.0125 Val Acc: 0.0141
Epoch [3/10] Train Loss: 4.6337 Train Acc: 0.0108 Val Acc: 0.0125
Epoch [4/10] Train Loss: 4.6337 Train Acc: 0.0139 Val Acc: 0.0141
Epoch [5/10] Train Loss: 4.6337 Train Acc: 0.0133 Val Acc: 0.0125
Epoch [6/10] Train Loss: 4.6338 Train Acc: 0.0125 Val Acc: 0.0141
Epoch [7/10] Train Loss: 4.6338 Train Acc: 0.0130 Val Acc: 0.0125
Epoch [8/10] Train Loss: 4.6330 Train Acc: 0.0123 Val Acc: 0.0125
Epoch [9/10] Train Loss: 4.6334 Train Acc: 0.0112 Val Acc: 0.0141
Epoch [10/10] Train Loss: 4.6328 Train Acc: 0.0135 Val Acc: 0.0141


In [38]:

# 6️⃣ Testing (After Training)
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_acc = test_correct / test_total
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.0103
